# Fundamentals 12 - Multi-Agentic Graph API

Graph multi-agente con nodos deterministas y un nodo LM opcional. El graph coordina estado; `runtime(provider="auto")` decide backend para el agente LM.


In [ ]:
import importlib.util
from typing import TypedDict
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=8, max_turns=8)
local_runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.AgenticSystem(model=lm_runtime.model_id or "python-runtime", region=lm_runtime.region_name or "local", runtime=lm_runtime)
toolkit.show({"local_runtime": local_runtime.describe(), "lm_runtime": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


## 1) Tools, agentes y estado


## Parámetros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42

@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}

@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}

policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solver = toolkit.agent(name="graph_solver", instructions="Resuelve n?meros estructurados.", tools=[solve_arithmetic], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic")), policy=policy)
judge = toolkit.agent(name="graph_judge", instructions="Valida resultado.", tools=[judge_result], engine="python-runtime", runtime=local_runtime, contract=toolkit.AgentContract(must_call=["judge_result"], tool_expectation=toolkit.expect.exactly("judge_result")), policy=policy)
reviewer = system.agent(name="graph_lm_reviewer", instructions="Resume riesgos sin cambiar el resultado.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))

class GraphState(TypedDict, total=False):
    prompt: str
    numbers: list[int]
    procedure: list[str]
    result: float
    judge: dict
    lm_review: str | None

toolkit.show({"agents": [solver.info(), judge.info(), reviewer.info()]})


## 2) Nodos del graph


In [ ]:
def solve_node(state: GraphState) -> GraphState:
    result = solver.run({"tool": "solve_arithmetic", "input": {"numbers": state["numbers"]}})
    return {**state, "procedure": result.data["procedure"], "result": result.data["result"], "solve_result": result}

def judge_node(state: GraphState) -> GraphState:
    result = judge.run({"tool": "judge_result", "input": {"result": state["result"], "expected": EXPECTED}})
    return {**state, "judge": result.data, "judge_result": result}

def review_node(state: GraphState) -> GraphState:
    if not lm_available:
        return {**state, "lm_review": None, "review_result": None}
    result = reviewer.run(str({"procedure": state["procedure"], "result": state["result"], "judge": state["judge"]}))
    return {**state, "lm_review": result.text, "review_result": result}

nodes = {"solve": solve_node, "judge": judge_node, "review": review_node}
edges = [("START", "solve"), ("solve", "judge"), ("judge", "review"), ("review", "END")]
toolkit.show({"nodes": list(nodes), "edges": edges})

## 3) Ejecutar con LangGraph si esta disponible; fallback local si no


In [ ]:
initial_state: GraphState = {"prompt": USER_PROMPT, "numbers": NUMBERS}
if importlib.util.find_spec("langgraph"):
    graph = toolkit.graph(state=GraphState, nodes=nodes, edges=edges, engine="langgraph", name="fundamentals_multi_agentic_graph")
    final_state = graph.run(initial_state)
    framework = "langgraph"
else:
    final_state = initial_state
    for node in [solve_node, judge_node, review_node]:
        final_state = node(final_state)
    framework = "local-state-pipeline"

payload = {
    "procedimiento": final_state["procedure"],
    "resultado_final": final_state["result"],
    "judge": final_state["judge"],
    "lm_review": final_state.get("lm_review"),
}
result = toolkit.compose_result(
    text="Graph multi-agente ejecutado.",
    data=payload,
    results=[final_state.get("solve_result"), final_state.get("judge_result"), final_state.get("review_result")],
    mode="multi-agentic-graph",
    framework=framework,
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.multi_agentic_graph", question=USER_PROMPT, goal="Explicar nodos, edges y estado final.")
toolkit.show({"framework": framework, "final_state_keys": list(final_state)})
toolkit.human_result(result, title="Human result - Multi-Agentic Graph", pretty=PRETTY, show_lineage=True, lineage=lineage)

## Coverage API


In [ ]:
toolkit.show({"notebook": "12_multi_agentic_graph_api.ipynb", "api_coverage": ["toolkit.graph", "nodes", "edges", "runtime(provider='auto')", "python-runtime nodes", "compose_result", "RunResult.lineage"]})


## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `toolkit.graph`: Coordinacion multi-agente como graph.
- `agent_node`: Adaptacion de agentes a nodos.
- `GraphStateOutput`: Estado compartido del graph.
- `toolkit.compose_result`: Resultado final auditable del graph.

